# 05 - Preprocessing

### Notebook Overview

Prepare the featured dataset for model training: encode categorical variables, address multicollinearity, scale numeric features, and split into train/test sets.

**Input:** `../data/car-price-featured.csv`  
**Output:** `../data/x-train.csv`, `../data/x-test.csv`, `../data/y-train.csv`, `../data/y-test.csv`

### 1 - Imports

In [1]:
# Core Libraries
import pandas as pd
import numpy as np

# Model building
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 2 - Load Data

In [2]:
df = pd.read_csv("../data/car-price-featured.csv")
print(f"Starting shape: {df.shape}")
df.dtypes

Starting shape: (205, 24)


symboling             int64
fueltype                str
aspiration              str
doornumber            int64
carbody                 str
drivewheel              str
wheelbase           float64
carlength           float64
carwidth            float64
carheight           float64
curbweight            int64
enginetype              str
cylindernumber        int64
enginesize            int64
fuelsystem              str
boreratio           float64
stroke              float64
compressionratio    float64
horsepower            int64
peakrpm               int64
price               float64
avg_mpg             float64
hp_per_weight       float64
brand_tier              str
dtype: object

### 3 - Encode Categorical Variables

Models need numeric inputs. How we encode depends on the variable type:

- **Binary categories** (`fueltype`, `aspiration`): label encode (0/1). With only two values, one column is sufficient.
- **Ordinal categories** (`brand_tier`): map to ordered integers. Budget < mid < premium has a meaningful order.
- **Nominal categories** (`carbody`, `drivewheel`, `enginetype`): one-hot encode. These have no natural order, so we create a binary column per value. We drop the first column to avoid the "dummy variable trap" (perfect multicollinearity where one column is fully determined by the others).

**Dropped before encoding:**
- `fuelsystem` — every diesel in this dataset uses `idi` injection and no gas car does, so `fuelsystem_idi` would be a perfect duplicate of `fueltype` after one-hot encoding. Several of its 8 categories also contain fewer than 10 cars, so the added dummies would amplify noise more than signal on a 205-row dataset. With a larger dataset, fuelsystem could be reintroduced to capture injection-type distinctions within gas cars.

In [3]:
# Binary encoding
df["fueltype"] = df["fueltype"].map({"gas": 0, "diesel": 1})
df["aspiration"] = df["aspiration"].map({"std": 0, "turbo": 1})

print("Binary encoded:")
print(f"  fueltype: {df['fueltype'].unique()}")
print(f"  aspiration: {df['aspiration'].unique()}")

Binary encoded:
  fueltype: [0 1]
  aspiration: [0 1]


In [4]:
# Ordinal encoding
df["brand_tier"] = df["brand_tier"].map({"budget": 0, "mid": 1, "premium": 2})

print("Ordinal encoded:")
print(f"  brand_tier: {sorted(df['brand_tier'].unique())}")

Ordinal encoded:
  brand_tier: [np.int64(0), np.int64(1), np.int64(2)]


In [5]:
# Drop fuelsystem (overlaps with fueltype + low per-category sample size; see markdown)
df = df.drop(columns=["fuelsystem"])

# One-hot encoding (drop_first=True to avoid dummy variable trap)
nominal_cols = ["carbody", "drivewheel", "enginetype"]
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

print(f"Shape after one-hot encoding: {df.shape}")

Shape after one-hot encoding: (205, 32)


### 4 - Address Multicollinearity

EDA identified clusters of highly correlated features (|r| > 0.8). For tree-based models this is harmless, but for linear regression correlated features make coefficients unstable and hard to interpret.

Strategy: from each correlated cluster, keep the feature with the strongest correlation to price and drop the rest.

| Cluster | Keep | Drop | Reason |
|---------|------|------|--------|
| Size (carlength, carwidth, curbweight, wheelbase) | `curbweight` (r=0.84) | `carlength`, `wheelbase` | curbweight has strongest price correlation; carwidth kept as it's less redundant (r=0.76) |
| Engine (enginesize, cylindernumber, horsepower) | `enginesize` (r=0.87) | `cylindernumber` | enginesize is most predictive; horsepower kept as it captures a different aspect (power vs displacement) |

In [6]:
drop_multicollinear = ["carlength", "wheelbase", "cylindernumber"]
df = df.drop(columns=drop_multicollinear)

print(f"Dropped {len(drop_multicollinear)} multicollinear features: {drop_multicollinear}")
print(f"Shape: {df.shape}")

Dropped 3 multicollinear features: ['carlength', 'wheelbase', 'cylindernumber']
Shape: (205, 29)


### 5 - Train/Test Split

We split before scaling to prevent data leakage. If we scaled first, the test set's statistics would influence the scaler, giving the model a subtle peek at test data.

With only 205 rows, we use an 80/20 split (164 train, 41 test). A fixed `random_state` ensures reproducibility.

In [7]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (164, 28)
X_test:  (41, 28)
y_train: (164,)
y_test:  (41,)


Also save an unscaled copy of the split for **notebook 08** (white-box model interpretation). The interpretation LR and DT need raw units so coefficients and tree thresholds read naturally.

In [8]:
X_train.to_csv("../data/x-train-unscaled.csv", index=False)
X_test.to_csv("../data/x-test-unscaled.csv", index=False)
print("Saved: x-train-unscaled.csv, x-test-unscaled.csv")

Saved: x-train-unscaled.csv, x-test-unscaled.csv


### 6 - Scale Numeric Features

StandardScaler transforms each feature to have mean=0 and std=1. This matters for linear regression (where coefficients are sensitive to feature scale) and helps gradient-based methods converge faster. Tree-based models don't need it, but it doesn't hurt them either.

We fit the scaler on training data only, then transform both train and test.

In [9]:
# Identify numeric columns to scale (exclude binary and one-hot encoded columns)
binary_and_onehot = ["fueltype", "aspiration", "brand_tier"] + [
    col for col in X_train.columns if col.startswith(("carbody_", "drivewheel_", "enginetype_", "fuelsystem_"))
]
numeric_to_scale = [col for col in X_train.columns if col not in binary_and_onehot]

print(f"Columns to scale ({len(numeric_to_scale)}): {numeric_to_scale}")
print(f"Columns left as-is ({len(binary_and_onehot)}): {binary_and_onehot}")

Columns to scale (13): ['symboling', 'doornumber', 'carwidth', 'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'avg_mpg', 'hp_per_weight']
Columns left as-is (15): ['fueltype', 'aspiration', 'brand_tier', 'carbody_hardtop', 'carbody_hatchback', 'carbody_sedan', 'carbody_wagon', 'drivewheel_fwd', 'drivewheel_rwd', 'enginetype_dohcv', 'enginetype_l', 'enginetype_ohc', 'enginetype_ohcf', 'enginetype_ohcv', 'enginetype_rotor']


In [10]:
scaler = StandardScaler()
X_train[numeric_to_scale] = scaler.fit_transform(X_train[numeric_to_scale])
X_test[numeric_to_scale] = scaler.transform(X_test[numeric_to_scale])

print("Scaling complete. Training set sample means (should be ~0):")
print(X_train[numeric_to_scale].mean().round(2).to_string())

Scaling complete. Training set sample means (should be ~0):
symboling           0.0
doornumber         -0.0
carwidth            0.0
carheight          -0.0
curbweight         -0.0
enginesize         -0.0
boreratio           0.0
stroke             -0.0
compressionratio   -0.0
horsepower         -0.0
peakrpm            -0.0
avg_mpg            -0.0
hp_per_weight      -0.0


### 7 - Verify Final State

In [11]:
print(f"Final feature count: {X_train.shape[1]}")
print("\nAll columns:")
print(X_train.dtypes.to_string())
print(f"\nNull values - train: {X_train.isnull().sum().sum()}, test: {X_test.isnull().sum().sum()}")

Final feature count: 28

All columns:
symboling            float64
fueltype               int64
aspiration             int64
doornumber           float64
carwidth             float64
carheight            float64
curbweight           float64
enginesize           float64
boreratio            float64
stroke               float64
compressionratio     float64
horsepower           float64
peakrpm              float64
avg_mpg              float64
hp_per_weight        float64
brand_tier             int64
carbody_hardtop        int64
carbody_hatchback      int64
carbody_sedan          int64
carbody_wagon          int64
drivewheel_fwd         int64
drivewheel_rwd         int64
enginetype_dohcv       int64
enginetype_l           int64
enginetype_ohc         int64
enginetype_ohcf        int64
enginetype_ohcv        int64
enginetype_rotor       int64

Null values - train: 0, test: 0


In [12]:
X_train.head()

,symboling,fueltype,aspiration,doornumber,carwidth,carheight,curbweight,enginesize,boreratio,stroke,...,carbody_sedan,carbody_wagon,drivewheel_fwd,drivewheel_rwd,enginetype_dohcv,enginetype_l,enginetype_ohc,enginetype_ohcf,enginetype_ohcv,enginetype_rotor
66,-0.727380,1,0,0.917985,0.083817,0.276779,0.281287,0.188010,0.405836,1.215101,...,1,0,0,1,0,0,1,0,0,0
111,-0.727380,0,0,0.917985,1.170268,1.207269,1.016045,-0.158993,0.517384,-3.360166,...,1,0,0,1,0,1,0,0,0,0
153,-0.727380,0,0,0.917985,-1.097108,2.178216,-0.541642,-0.852999,-1.007108,-0.709667,...,0,1,1,0,0,0,1,0,0,0
96,0.078636,0,0,0.917985,-1.002634,0.317235,-1.147082,-0.729070,-0.635281,0.110726,...,1,0,1,0,0,0,1,0,0,0
38,-0.727380,0,0,-1.089342,-0.341316,-0.168238,-0.524008,-0.406852,-0.635281,1.025779,...,0,0,1,0,0,0,1,0,0,0


### 8 - Save Preprocessed Data

In [13]:
X_train.to_csv("../data/x-train.csv", index=False)
X_test.to_csv("../data/x-test.csv", index=False)
y_train.to_csv("../data/y-train.csv", index=False)
y_test.to_csv("../data/y-test.csv", index=False)

print("Saved: x-train.csv, x-test.csv, y-train.csv, y-test.csv")

Saved: x-train.csv, x-test.csv, y-train.csv, y-test.csv


### 9 - Summary

**Encoding:**
- Binary: `fueltype` (gas=0, diesel=1), `aspiration` (std=0, turbo=1)
- Ordinal: `brand_tier` (budget=0, mid=1, premium=2)
- One-hot: `carbody`, `drivewheel`, `enginetype` with `drop_first=True`.

**Dropped before encoding:**
- `fuelsystem` — overlap with `fueltype` via the `idi` category + low per-category sample sizes.

**Multicollinearity (numeric):** dropped `carlength`, `wheelbase`, `cylindernumber` (redundant with `curbweight`, `carwidth`, `enginesize`).

**Split:** 80/20 train/test (164 / 41 rows), random_state=42.

**Scaling:** StandardScaler on continuous features, fit on training data only.

**Final shape:** ready for model training.